# 08 — LoRA Fine-tuning + ControlNet + IP-Adapter (Local)

Train LoRA **r=16** tren UNet `runwayml/stable-diffusion-inpainting`, sau do gan
**ControlNet Canny** + **IP-Adapter Plus** va danh gia tren cung **1,998 anh test**
voi `full-eval.ipynb` de so sanh cong bang.

**Pipeline:** `x_occ -> LoRA-UNet + ControlNet(Canny) + IP-Adapter -> x_hat`

**Metrics:** L1, L2, ICP, SS (Yan et al. 2019) + PSNR, SSIM, LPIPS, FID

**Data:** `data/synthetic_occ/` (local)


In [ ]:
# CELL 1 — Dependencies
import sys
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers>=0.27.0', 'peft>=0.8.0', 'accelerate>=0.26.0',
    'transformers>=4.36.0', 'xformers',
    'scikit-image', 'lpips', 'clean-fid', 'opencv-python',
    'huggingface_hub', 'tqdm'], check=True)
print('Dependencies ready')


In [ ]:
# CELL 2 — Config & Paths (LOCAL)
import random, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path('.').resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SYNTH_DIR = PROJECT_ROOT / 'data' / 'synthetic_occ'
GT_DIR    = SYNTH_DIR / 'x_gt'
OCC_DIR   = SYNTH_DIR / 'x_occ'
MASK_DIR  = SYNTH_DIR / 'masks'
META_CSV  = SYNTH_DIR / 'metadata_synthetic_occ.csv'

LORA_OUT      = PROJECT_ROOT / 'outputs' / 'lora_weights' / 'r16_cn_ipa'
OUT_BASE      = PROJECT_ROOT / 'outputs' / 'lora_cn_ipa'
PRED_DIR      = OUT_BASE / 'x_hat'
EVAL_GT_DIR   = OUT_BASE / 'fid_gt'
EVAL_PRED_DIR = OUT_BASE / 'fid_pred'
REPORT_DIR    = OUT_BASE / 'reports'
for d in [LORA_OUT, PRED_DIR, EVAL_GT_DIR, EVAL_PRED_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16 if DEVICE == 'cuda' else torch.float32

SD_MODEL_ID = 'runwayml/stable-diffusion-inpainting'
CN_MODEL_ID = 'lllyasviel/control_v11p_sd15_canny'
IP_REPO     = 'h94/IP-Adapter'
IP_WEIGHT   = 'models/ip-adapter-plus_sd15.bin'

LORA_RANK    = 16
LORA_ALPHA   = 16
LORA_DROPOUT = 0.05

LEARNING_RATE       = 1e-4
MAX_TRAIN_STEPS     = 2000
TRAIN_BATCH_SIZE    = 1
GRAD_ACCUM_STEPS    = 8
LR_WARMUP_STEPS     = 200
SNR_GAMMA           = 5.0
NOISE_OFFSET        = 0.1
CHECKPOINTING_STEPS = 500
VALIDATION_STEPS    = 250
VAL_SIZE            = 50
EARLY_STOP_PATIENCE = 6
MAX_CKPT_KEEP       = 3
LOSS_MASK_WEIGHT    = 0.8
LOSS_FULL_WEIGHT    = 0.2

PROMPT_POOL = [
    'a car, realistic, high quality, detailed',
    'a photo of a car, sharp, clear, complete',
    'automobile, complete, undamaged, high resolution',
    'a vehicle, realistic appearance, full body visible',
    'car exterior, clean, detailed, photorealistic',
]

PROMPT     = 'a car, realistic, high quality, detailed, complete, no occlusion'
NEG_PROMPT = 'blurry, distorted, artifacts, extra car, duplicate'
NUM_STEPS  = 20
GUIDANCE   = 7.5

# SKIP_GRID_SEARCH=True dung gia tri best tu full-eval (cn=0.6, ip=0.5)
# Dat False neu muon tim lai combo toi uu cho LoRA pipeline
SKIP_GRID_SEARCH = True
BEST_CN          = 0.60
BEST_IP          = 0.50
CN_SCALE_GRID    = [0.15, 0.30, 0.45, 0.60, 0.75]
IP_SCALE_GRID    = [0.3, 0.5, 0.7]
GRID_SAMPLE      = 40

CANNY_LOW      = 80
CANNY_HIGH     = 150
MASK_DILATE_PX = 5

BIN_EDGES  = [(0.20, 0.40), (0.40, 0.60), (0.60, 0.80)]
BIN_LABELS = ['20-40%', '40-60%', '60-80%']

assert SYNTH_DIR.exists(), f'Dataset khong tim thay: {SYNTH_DIR}'
assert META_CSV.exists(),  f'Metadata khong tim thay: {META_CSV}'

meta = pd.read_csv(META_CSV)
meta.columns = meta.columns.str.strip().str.lower()
print(f'Dataset    : {len(meta):,} images')
print(f'Device     : {DEVICE} | dtype={DTYPE}')
print(f'LoRA       : r={LORA_RANK}, alpha={LORA_ALPHA}, steps={MAX_TRAIN_STEPS}')
print(f'Models     : {SD_MODEL_ID}')
print(f'ControlNet : {CN_MODEL_ID}')


In [ ]:
# CELL 3 — Train / Val / Test Split
# Test: 666/bin x 3 bins, seed=42 -- khop voi full-eval.ipynb
_per_bin = 2000 // len(BIN_EDGES)
_bin_dfs = []
for lo, hi in BIN_EDGES:
    sub = meta[(meta['occlusion_ratio'] >= lo) & (meta['occlusion_ratio'] < hi)]
    _bin_dfs.append(sub.sample(min(_per_bin, len(sub)), random_state=SEED))
test_df    = pd.concat(_bin_dfs).sample(frac=1, random_state=SEED).reset_index(drop=True)
test_stems = set(test_df['stem'].tolist())

non_test   = meta[~meta['stem'].isin(test_stems)].reset_index(drop=True)
val_meta   = non_test.sample(min(VAL_SIZE, len(non_test)), random_state=SEED).reset_index(drop=True)
train_meta = non_test[~non_test['stem'].isin(set(val_meta['stem']))].reset_index(drop=True)

print(f'Train : {len(train_meta):,}')
print(f'Val   : {len(val_meta):,}')
print(f'Test  : {len(test_df):,}  (shared voi full-eval -- 666/bin x 3 bins)')
for (lo,hi), lbl in zip(BIN_EDGES, BIN_LABELS):
    n = len(test_df[(test_df['occlusion_ratio']>=lo)&(test_df['occlusion_ratio']<hi)])
    print(f'  {lbl}: {n} anh')


In [ ]:
# CELL 4 — Dataset Class
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF
from PIL import Image

class CarInpaintDataset(Dataset):
    def __init__(self, meta_df, gt_dir, occ_dir, mask_dir, tokenizer,
                 prompt_pool=None, size=512, augment=True):
        self.meta        = meta_df.reset_index(drop=True)
        self.gt_dir      = Path(gt_dir)
        self.occ_dir     = Path(occ_dir)
        self.mask_dir    = Path(mask_dir)
        self.tokenizer   = tokenizer
        self.prompt_pool = prompt_pool or ['a car, realistic, high quality, detailed']
        self.size        = size
        self.augment     = augment
        self.normalize   = transforms.Normalize([0.5]*3, [0.5]*3)
        self.to_tensor   = transforms.ToTensor()
        self.jitter      = transforms.ColorJitter(
            brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02
        ) if augment else None

    def __len__(self):
        return len(self.meta)

    def _load(self, row):
        gt   = Image.open(self.gt_dir   / row['x_gt']).convert('RGB')
        occ  = Image.open(self.occ_dir  / row['x_occ']).convert('RGB')
        mname = row.get('mask', f"{row['stem']}.png")
        mask = Image.open(self.mask_dir / mname).convert('L')
        gt   = gt.resize((self.size, self.size), Image.LANCZOS)
        occ  = occ.resize((self.size, self.size), Image.LANCZOS)
        mask = mask.resize((self.size, self.size), Image.NEAREST)
        return gt, occ, mask

    def _augment(self, gt, occ, mask):
        if torch.rand(1).item() < 0.5:
            gt = TF.hflip(gt); occ = TF.hflip(occ); mask = TF.hflip(mask)
        if self.jitter:
            fn_idx, b, c, s, h = self.jitter.get_params(
                self.jitter.brightness, self.jitter.contrast,
                self.jitter.saturation, self.jitter.hue)
            for fn, val in [(TF.adjust_brightness, float(b)), (TF.adjust_contrast, float(c)),
                            (TF.adjust_saturation, float(s)), (TF.adjust_hue, float(h))]:
                gt = fn(gt, val); occ = fn(occ, val)
        if torch.rand(1).item() < 0.3:
            scale    = random.uniform(0.9, 1.1)
            new_size = max(int(self.size * scale), self.size)
            gt   = TF.resize(gt,   new_size, Image.LANCZOS)
            occ  = TF.resize(occ,  new_size, Image.LANCZOS)
            mask = TF.resize(mask, new_size, Image.NEAREST)
            gt   = TF.center_crop(gt,   self.size)
            occ  = TF.center_crop(occ,  self.size)
            mask = TF.center_crop(mask, self.size)
        return gt, occ, mask

    def __getitem__(self, idx):
        row = self.meta.iloc[idx]
        gt, occ, mask = self._load(row)
        if self.augment:
            gt, occ, mask = self._augment(gt, occ, mask)
        gt_t     = self.normalize(self.to_tensor(gt))
        mask_t   = (self.to_tensor(mask) > 0.5).float()
        masked_t = self.normalize(self.to_tensor(occ)) * (1.0 - mask_t)
        prompt = random.choice(self.prompt_pool)
        enc = self.tokenizer(
            prompt, padding='max_length', max_length=self.tokenizer.model_max_length,
            truncation=True, return_tensors='pt')
        return {'pixel_values': gt_t, 'masked_image': masked_t,
                'mask': mask_t, 'input_ids': enc.input_ids[0],
                'occlusion_ratio': float(row.get('occlusion_ratio', 0.0))}

print('CarInpaintDataset defined.')


In [ ]:
# CELL 5 — Load SD1.5 + Inject LoRA r=16
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
from transformers import CLIPTokenizer, CLIPTextModel
from peft import LoraConfig, get_peft_model

tokenizer    = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder='tokenizer')
text_encoder = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder='text_encoder')
vae          = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder='vae')
unet         = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder='unet')
noise_sched  = DDPMScheduler.from_pretrained(SD_MODEL_ID, subfolder='scheduler')

vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

lora_config = LoraConfig(
    r               = LORA_RANK,
    lora_alpha      = LORA_ALPHA,
    target_modules  = [
        'to_q', 'to_k', 'to_v', 'to_out.0',
        'proj_in', 'proj_out',
        'ff.net.0.proj', 'ff.net.2',
    ],
    lora_dropout     = LORA_DROPOUT,
    bias             = 'none',
    init_lora_weights= 'gaussian',
)
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()
unet.enable_gradient_checkpointing()

unet.to(DEVICE, dtype=torch.float32)
vae.to(DEVICE,  dtype=DTYPE)
text_encoder.to(DEVICE, dtype=DTYPE)

train_dataset = CarInpaintDataset(
    train_meta, GT_DIR, OCC_DIR, MASK_DIR, tokenizer,
    prompt_pool=PROMPT_POOL, augment=True)
val_dataset = CarInpaintDataset(
    val_meta, GT_DIR, OCC_DIR, MASK_DIR, tokenizer,
    prompt_pool=PROMPT_POOL, augment=False)
train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE,
    shuffle=True, num_workers=2, pin_memory=(DEVICE=='cuda'), drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=TRAIN_BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=(DEVICE=='cuda'))

sample_b = next(iter(train_loader))
print(f'pixel_values : {sample_b["pixel_values"].shape}')
print(f'Train: {len(train_loader)} batches  |  Val: {len(val_loader)} batches')


In [ ]:
# CELL 6 — Training Loop
import shutil
from transformers import get_cosine_schedule_with_warmup
from tqdm.auto import tqdm

trainable_params_list = [p for p in unet.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params_list,
    lr=LEARNING_RATE, betas=(0.9, 0.999), weight_decay=1e-2, eps=1e-8)
lr_scheduler = get_cosine_schedule_with_warmup(optimizer,
    num_warmup_steps=LR_WARMUP_STEPS, num_training_steps=MAX_TRAIN_STEPS)
scaler         = torch.amp.GradScaler('cuda') if DEVICE == 'cuda' else None
alphas_cumprod = noise_sched.alphas_cumprod.to(DEVICE)

def compute_snr(t):
    a = alphas_cumprod[t].sqrt()
    b = (1.0 - alphas_cumprod[t]).sqrt()
    return (a / b) ** 2

def forward_pass(batch):
    gt_pv  = batch['pixel_values'].to(DEVICE, dtype=DTYPE)
    mask   = batch['mask'].to(DEVICE, dtype=DTYPE)
    masked = batch['masked_image'].to(DEVICE, dtype=DTYPE)
    ids    = batch['input_ids'].to(DEVICE)
    with torch.no_grad():
        latents        = vae.encode(gt_pv).latent_dist.sample() * vae.config.scaling_factor
        masked_latents = vae.encode(masked).latent_dist.sample() * vae.config.scaling_factor
        mask_latent    = F.interpolate(mask, size=latents.shape[-2:], mode='nearest')
        encoder_hidden = text_encoder(ids)[0]
    noise = torch.randn_like(latents)
    if NOISE_OFFSET > 0:
        noise = noise + NOISE_OFFSET * torch.randn(
            latents.shape[0], latents.shape[1], 1, 1, device=DEVICE, dtype=DTYPE)
    bsz = latents.shape[0]
    t   = torch.randint(0, noise_sched.config.num_train_timesteps, (bsz,), device=DEVICE, dtype=torch.long)
    noisy = noise_sched.add_noise(latents, noise, t)
    model_input = torch.cat([noisy, mask_latent, masked_latents], dim=1)
    noise_pred  = unet(model_input.float(), t, encoder_hidden_states=encoder_hidden.float()).sample
    snr    = compute_snr(t).float()
    weight = torch.clamp(snr, max=SNR_GAMMA) / snr if SNR_GAMMA > 0 else torch.ones(bsz, device=DEVICE)
    diff   = (noise_pred - noise.float()) ** 2
    loss   = ((LOSS_MASK_WEIGHT * (diff * mask_latent).mean(dim=[1,2,3]) +
               LOSS_FULL_WEIGHT * diff.mean(dim=[1,2,3])) * weight).mean()
    return loss

def compute_val_loss(val_loader, fixed_seed=42):
    unet.eval()
    total = 0.0
    with torch.no_grad():
        for i, vb in enumerate(val_loader):
            torch.manual_seed(fixed_seed + i)
            if DEVICE == 'cuda':
                with torch.amp.autocast('cuda'):
                    total += forward_pass(vb).item()
            else:
                total += forward_pass(vb).item()
    return total / len(val_loader)

global_step        = 0
running_loss       = 0.0
train_losses       = []
val_losses         = []
best_val_loss      = float('inf')
early_stop_counter = 0
saved_checkpoints  = []

unet.train()
data_iter = iter(train_loader)
optimizer.zero_grad()
t0_train = time.time()
pbar = tqdm(range(MAX_TRAIN_STEPS), desc=f'LoRA r={LORA_RANK}', dynamic_ncols=True)

for step in pbar:
    accum_loss = 0.0
    for _ in range(GRAD_ACCUM_STEPS):
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            batch = next(data_iter)
        if DEVICE == 'cuda' and scaler is not None:
            with torch.amp.autocast('cuda'):
                loss = forward_pass(batch) / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
        else:
            loss = forward_pass(batch) / GRAD_ACCUM_STEPS
            loss.backward()
        accum_loss += loss.item()
    if DEVICE == 'cuda' and scaler is not None:
        scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(trainable_params_list, max_norm=1.0)
    if DEVICE == 'cuda' and scaler is not None:
        scaler.step(optimizer); scaler.update()
    else:
        optimizer.step()
    lr_scheduler.step()
    optimizer.zero_grad()
    global_step  += 1
    running_loss += accum_loss * GRAD_ACCUM_STEPS
    if global_step % 50 == 0:
        avg = running_loss / 50
        train_losses.append({'step': global_step, 'loss': avg})
        pbar.set_postfix({'loss': f'{avg:.4f}', 'lr': f'{lr_scheduler.get_last_lr()[0]:.2e}'})
        running_loss = 0.0
    if global_step % CHECKPOINTING_STEPS == 0:
        ckpt_dir = LORA_OUT / f'checkpoint-{global_step}'
        unet.save_pretrained(str(ckpt_dir))
        saved_checkpoints.append(str(ckpt_dir))
        print(f'  [Step {global_step}] Checkpoint -> {ckpt_dir.name}')
        if len(saved_checkpoints) > MAX_CKPT_KEEP:
            oldest = saved_checkpoints.pop(0)
            shutil.rmtree(oldest, ignore_errors=True)
    if global_step % VALIDATION_STEPS == 0:
        avg_val = compute_val_loss(val_loader, fixed_seed=SEED)
        val_losses.append({'step': global_step, 'val_loss': avg_val})
        elapsed = (time.time() - t0_train) / 60
        print(f'  [Step {global_step:4d}] val_loss={avg_val:.4f} | {elapsed:.1f} min')
        if avg_val < best_val_loss:
            best_val_loss = avg_val; early_stop_counter = 0
            unet.save_pretrained(str(LORA_OUT / 'best'))
            print(f'  Best saved (val={best_val_loss:.4f})')
        else:
            early_stop_counter += 1
            print(f'  No improve ({early_stop_counter}/{EARLY_STOP_PATIENCE})')
            if early_stop_counter >= EARLY_STOP_PATIENCE:
                print(f'  Early stop at step {global_step}.')
                break
        unet.train()

unet.save_pretrained(str(LORA_OUT / 'final'))
total_time = (time.time() - t0_train) / 60
print(f'Done: {global_step} steps | {total_time:.1f} min | best_val={best_val_loss:.4f}')


In [ ]:
# CELL 7 — Loss Curves + Save Config
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'LoRA r={LORA_RANK} — mixed loss {LOSS_MASK_WEIGHT:.0%}/{LOSS_FULL_WEIGHT:.0%}',
             fontweight='bold')
if train_losses:
    df_tr = pd.DataFrame(train_losses)
    axes[0].plot(df_tr['step'], df_tr['loss'], color='#4C72B0')
    axes[0].set(title='Train Loss', xlabel='Steps', ylabel='Loss'); axes[0].grid(alpha=0.3)
if val_losses:
    df_val = pd.DataFrame(val_losses)
    axes[1].plot(df_val['step'], df_val['val_loss'], color='#DD8452', marker='o', markersize=5)
    axes[1].axhline(best_val_loss, linestyle='--', color='gray', label=f'best={best_val_loss:.4f}')
    axes[1].set(title='Validation Loss', xlabel='Steps'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(LORA_OUT / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

cfg = {
    'base_model': SD_MODEL_ID, 'lora_rank': LORA_RANK, 'lora_alpha': LORA_ALPHA,
    'target_modules': ['to_q','to_k','to_v','to_out.0','proj_in','proj_out','ff.net.0.proj','ff.net.2'],
    'loss_mask_weight': LOSS_MASK_WEIGHT, 'loss_full_weight': LOSS_FULL_WEIGHT,
    'max_steps': MAX_TRAIN_STEPS, 'actual_steps': global_step,
    'learning_rate': LEARNING_RATE, 'batch_size': TRAIN_BATCH_SIZE,
    'grad_accum': GRAD_ACCUM_STEPS, 'snr_gamma': SNR_GAMMA,
    'best_val_loss': best_val_loss, 'train_size': len(train_meta), 'val_size': len(val_meta),
}
with open(LORA_OUT / 'lora_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)
print(f'Config saved -> {LORA_OUT}/lora_config.json')
print(f'Best  -> {LORA_OUT}/best')
print(f'Final -> {LORA_OUT}/final')


In [ ]:
# CELL 8 — Load LoRA + ControlNet + IP-Adapter Pipeline
import cv2
from PIL import Image
from peft import PeftModel
from diffusers import (
    StableDiffusionControlNetInpaintPipeline,
    ControlNetModel,
    DPMSolverMultistepScheduler,
)

print('Loading ControlNet ...')
controlnet = ControlNetModel.from_pretrained(CN_MODEL_ID, torch_dtype=DTYPE)

print('Loading SD1.5 Inpainting pipeline ...')
pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
    SD_MODEL_ID, controlnet=controlnet, torch_dtype=DTYPE,
    safety_checker=None, requires_safety_checker=False)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

best_ckpt = LORA_OUT / 'best'
if not best_ckpt.exists():
    best_ckpt = LORA_OUT / 'final'
    print('  best khong tim thay -- dung final')
print(f'Merging LoRA from {best_ckpt.name} ...')
pipe.unet = PeftModel.from_pretrained(pipe.unet, str(best_ckpt))
pipe.unet = pipe.unet.merge_and_unload()
print('LoRA merged into UNet')

pipe = pipe.to(DEVICE)
xformers_ok = False
try:
    pipe.enable_xformers_memory_efficient_attention()
    xformers_ok = True; print('xFormers: enabled')
except Exception:
    pipe.enable_attention_slicing('auto'); print('xFormers: fallback -> attention slicing')
pipe.vae.enable_slicing()

use_ip = False
try:
    pipe.load_ip_adapter(IP_REPO, subfolder='models', weight_name=IP_WEIGHT)
    use_ip = True; print('IP-Adapter: loaded OK')
except Exception as e:
    print(f'IP-Adapter: FAILED ({e}) -- running without IP-Adapter')

print(f'Pipeline ready | LoRA r={LORA_RANK} merged | IP-Adapter={use_ip} | xFormers={xformers_ok}')
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU VRAM: {alloc:.2f}/{total:.1f} GiB')


In [ ]:
# CELL 9 — Inference Helpers

def extract_canny_masked(image_pil, mask_pil,
                          low=CANNY_LOW, high=CANNY_HIGH, dilate_px=MASK_DILATE_PX):
    img   = np.array(image_pil.convert('RGB'))
    gray  = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, low, high)
    mask_np = np.array(mask_pil.convert('L'))
    if dilate_px > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilate_px*2+1, dilate_px*2+1))
        mask_np = cv2.dilate(mask_np, k, iterations=1)
    edges[mask_np > 127] = 0
    return Image.fromarray(np.stack([edges]*3, axis=2))

def extract_visible_patch(image_pil, mask_pil):
    img  = np.array(image_pil.convert('RGB'))
    mask = np.array(mask_pil.convert('L'))
    vis  = img.copy(); vis[mask > 127] = 128
    return Image.fromarray(vis)

def inpaint(image, mask, prompt, cn_scale=BEST_CN, ip_scale=BEST_IP, seed=SEED):
    canny   = extract_canny_masked(image, mask)
    gen     = torch.Generator(device=DEVICE).manual_seed(seed)
    vis_ref = extract_visible_patch(image, mask)
    call_kwargs = dict(
        prompt=prompt, negative_prompt=NEG_PROMPT,
        image=image, control_image=canny, mask_image=mask,
        num_inference_steps=NUM_STEPS, guidance_scale=GUIDANCE,
        controlnet_conditioning_scale=cn_scale, generator=gen)
    if use_ip:
        pipe.set_ip_adapter_scale(ip_scale)
        call_kwargs['ip_adapter_image'] = vis_ref
    ctx = torch.autocast('cuda') if DEVICE == 'cuda' else torch.no_grad()
    with ctx:
        result = pipe(**call_kwargs).images[0]
    torch.cuda.empty_cache()
    return result

print('Helpers defined: extract_canny_masked / extract_visible_patch / inpaint')


In [ ]:
# CELL 10 — Preview 4 mau tu test set
import matplotlib.pyplot as plt

samples_vis = test_df.sample(4, random_state=SEED).reset_index(drop=True)
n_cols = 5 if use_ip else 4
fig, axes = plt.subplots(4, n_cols, figsize=(4*n_cols, 14))
for i, row in samples_vis.iterrows():
    occ = Image.open(OCC_DIR  / row['x_occ']).convert('RGB').resize((512,512))
    msk = Image.open(MASK_DIR / row['mask']).convert('L').resize((512,512))
    gt  = Image.open(GT_DIR   / row['x_gt']).convert('RGB').resize((512,512))
    can = extract_canny_masked(occ, msk)
    col = 0
    def show(ax, img, title, cmap=None):
        ax.imshow(img, cmap=cmap)
        if i == 0: ax.set_title(title, fontsize=9, fontweight='bold')
        ax.axis('off')
    show(axes[i,col], occ, 'x_occ'); col+=1
    show(axes[i,col], msk, 'Mask', 'gray'); col+=1
    show(axes[i,col], can, 'Canny'); col+=1
    if use_ip:
        show(axes[i,col], extract_visible_patch(occ,msk), 'Visible ref'); col+=1
    show(axes[i,col], gt, 'x_gt (GT)')
    axes[i,0].set_ylabel(f"occ={row['occlusion_ratio']:.2f}", fontsize=8)
plt.suptitle(f'Preview -- LoRA r={LORA_RANK} + CN + IPA', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# CELL 11 — Grid Search hoac Fixed Scales
from tqdm.auto import tqdm
from skimage.metrics import structural_similarity as ssim_fn, peak_signal_noise_ratio as psnr_fn
import lpips as lpips_lib

lpips_model = lpips_lib.LPIPS(net='alex').to(DEVICE).eval()

def pixel_l1(gt, pred, mask01):
    gt_f=gt.astype(np.float64)/255.; pr_f=pred.astype(np.float64)/255.; m=mask01.astype(bool)
    return float(np.mean(np.abs(gt_f[m]-pr_f[m]))) if m.sum()>0 else float(np.mean(np.abs(gt_f-pr_f)))

def pixel_l2(gt, pred, mask01):
    gt_f=gt.astype(np.float64)/255.; pr_f=pred.astype(np.float64)/255.; m=mask01.astype(bool)
    return float(np.mean((gt_f[m]-pr_f[m])**2)) if m.sum()>0 else float(np.mean((gt_f-pr_f)**2))

def masked_psnr(gt, pred, mask01):
    m=mask01.astype(bool)
    gt_m=gt[m].astype(np.float64); pr_m=pred[m].astype(np.float64)
    mse=np.mean((gt_m-pr_m)**2) if m.sum()>0 else np.mean((gt.astype(np.float64)-pred.astype(np.float64))**2)
    return float(10*np.log10(255.**2/mse)) if mse>0 else 100.0

def masked_ssim(gt, pred, mask01):
    _,smap=ssim_fn(gt.astype(np.float32)/255.,pred.astype(np.float32)/255.,
                   channel_axis=2,data_range=1.0,full=True)
    m=mask01.astype(bool)
    return float(smap[m].mean()) if m.sum()>0 else float(smap.mean())

def masked_lpips(gt, pred, mask01):
    m=mask01.astype(np.float32)[...,None]
    gt_t=torch.from_numpy((gt*m).astype(np.uint8)).permute(2,0,1).unsqueeze(0).float()/127.5-1.
    pr_t=torch.from_numpy((pred*m).astype(np.uint8)).permute(2,0,1).unsqueeze(0).float()/127.5-1.
    with torch.no_grad():
        return float(lpips_model(gt_t.to(DEVICE), pr_t.to(DEVICE)).item())

if SKIP_GRID_SEARCH:
    print(f'Grid search tat -- dung tu full-eval: cn_scale={BEST_CN}, ip_scale={BEST_IP}')
    grid_df = pd.DataFrame([{'cn_scale':BEST_CN,'ip_scale':BEST_IP,
                              'ssim_mean':None,'lpips_mean':None,'psnr_mean':None}])
else:
    grid_sample = test_df.sample(GRID_SAMPLE, random_state=SEED).reset_index(drop=True)
    ip_values   = IP_SCALE_GRID if use_ip else [0.0]
    grid_res    = []
    for cn_s in CN_SCALE_GRID:
        for ip_s in ip_values:
            ss,lp,ps=[],[],[]
            for _,r in tqdm(grid_sample.iterrows(), total=len(grid_sample), desc=f'cn={cn_s} ip={ip_s}'):
                occ=Image.open(OCC_DIR/r['x_occ']).convert('RGB').resize((512,512))
                msk=Image.open(MASK_DIR/r['mask']).convert('L').resize((512,512))
                gt=Image.open(GT_DIR/r['x_gt']).convert('RGB').resize((512,512))
                pred=inpaint(occ,msk,PROMPT,cn_scale=cn_s,ip_scale=ip_s)
                gt_np=np.array(gt);pr_np=np.array(pred);mk01=(np.array(msk)>127).astype(np.uint8)
                ss.append(masked_ssim(gt_np,pr_np,mk01))
                lp.append(masked_lpips(gt_np,pr_np,mk01))
                ps.append(masked_psnr(gt_np,pr_np,mk01))
            row_r={'cn_scale':cn_s,'ip_scale':ip_s,'ssim_mean':float(np.mean(ss)),
                   'lpips_mean':float(np.mean(lp)),'psnr_mean':float(np.mean(ps))}
            grid_res.append(row_r)
            print(f'  cn={cn_s} ip={ip_s} | PSNR={row_r["psnr_mean"]:.2f} LPIPS={row_r["lpips_mean"]:.4f}')
    grid_df = pd.DataFrame(grid_res)
    best_idx = int(grid_df['lpips_mean'].idxmin())
    BEST_CN  = float(grid_df.loc[best_idx,'cn_scale'])
    BEST_IP  = float(grid_df.loc[best_idx,'ip_scale'])
    grid_df.to_csv(REPORT_DIR/'grid_search.csv', index=False)
    print(f'Best combo: cn_scale={BEST_CN}  ip_scale={BEST_IP}')

print(f'Using: cn_scale={BEST_CN}  ip_scale={BEST_IP}')


In [ ]:
# CELL 12 — Full Inference tren 1,998 anh test
import time
from tqdm.auto import tqdm

print(f'Inference: {len(test_df):,} images | cn={BEST_CN} ip={BEST_IP} steps={NUM_STEPS}')
pred_records = []
t0 = time.time()

for _, r in tqdm(test_df.iterrows(), total=len(test_df), desc='Inference'):
    occ  = Image.open(OCC_DIR  / r['x_occ']).convert('RGB').resize((512,512))
    msk  = Image.open(MASK_DIR / r['mask']).convert('L').resize((512,512))
    gt   = Image.open(GT_DIR   / r['x_gt']).convert('RGB').resize((512,512))
    pred = inpaint(occ, msk, PROMPT, cn_scale=BEST_CN, ip_scale=BEST_IP, seed=SEED)
    fname = f"{r['stem']}.png"
    pred.save(PRED_DIR      / fname)
    gt.save  (EVAL_GT_DIR   / fname)
    pred.save(EVAL_PRED_DIR / fname)
    pred_records.append({
        'stem': r['stem'], 'x_gt': r['x_gt'], 'x_occ': r['x_occ'],
        'mask': r['mask'], 'occlusion_ratio': r['occlusion_ratio'],
        'bin_label': r.get('bin_label','?'), 'pred': fname})

elapsed = time.time() - t0
pred_df = pd.DataFrame(pred_records)
pred_df.to_csv(REPORT_DIR / 'pred_index.csv', index=False)
print(f'Done: {len(pred_df):,} images | {elapsed/60:.1f} min | ~{elapsed/max(len(pred_df),1):.1f}s/img')


In [ ]:
# CELL 13 — Compute All Metrics (L1, L2, ICP, SS, PSNR, SSIM, LPIPS, FID)
from cleanfid import fid as cleanfid
from torchvision import transforms
from torchvision.models import inception_v3
from torchvision.models.segmentation import deeplabv3_resnet101

print('Loading Inception V3 (ICP) ...')
try:
    from torchvision.models import Inception_V3_Weights
    inception_net = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1)
except (ImportError, AttributeError):
    inception_net = inception_v3(pretrained=True)
inception_net.eval().to(DEVICE)

IMAGENET_CAR_CLASSES = [407,436,511,627,656,705,717,734,751,779,817,820,868]
inception_tf = transforms.Compose([
    transforms.Resize(299), transforms.CenterCrop(299),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

def compute_icp(pred_pil, mask_np_gray):
    ys,xs = np.where(mask_np_gray > 127)
    if len(ys) == 0: return 0.0
    pad=16; h,w=mask_np_gray.shape
    y0=max(0,int(ys.min())-pad); y1=min(h,int(ys.max())+pad)
    x0=max(0,int(xs.min())-pad); x1=min(w,int(xs.max())+pad)
    crop = pred_pil.crop((x0,y0,x1,y1))
    if crop.width < 10 or crop.height < 10: return 0.0
    inp = inception_tf(crop.convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out    = inception_net(inp)
        logits = out.logits if hasattr(out,'logits') else out
        probs  = torch.softmax(logits, dim=1)[0].cpu()
    return float(probs[IMAGENET_CAR_CLASSES].sum())

print('Loading DeepLab V3 (SS) ...')
try:
    from torchvision.models.segmentation import DeepLabV3_ResNet101_Weights
    deeplab_net = deeplabv3_resnet101(weights=DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1)
except (ImportError, AttributeError):
    deeplab_net = deeplabv3_resnet101(pretrained=True)
deeplab_net.eval().to(DEVICE)
CAR_CLASS_IDX = 7
seg_tf = transforms.Compose([transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

def compute_ss(pred_pil, mask01):
    inp = seg_tf(pred_pil.convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        seg_out = deeplab_net(inp)['out'][0]
    pred_seg = seg_out.argmax(0).cpu().numpy().astype(np.uint8)
    pred_seg = cv2.resize(pred_seg,(512,512),interpolation=cv2.INTER_NEAREST)
    m = mask01.astype(bool)
    return float((pred_seg[m] == CAR_CLASS_IDX).mean()) if m.sum()>0 else 0.0

print('Computing per-image metrics ...')
from tqdm.auto import tqdm
metric_rows = []
for _, r in tqdm(pred_df.iterrows(), total=len(pred_df), desc='Metrics'):
    gt_bgr = cv2.imread(str(GT_DIR   / r['x_gt']))
    pr_bgr = cv2.imread(str(PRED_DIR / r['pred']))
    if gt_bgr is None or pr_bgr is None: continue
    gt_np  = cv2.resize(cv2.cvtColor(gt_bgr, cv2.COLOR_BGR2RGB),(512,512))
    pr_np  = cv2.resize(cv2.cvtColor(pr_bgr, cv2.COLOR_BGR2RGB),(512,512))
    mk_raw = cv2.imread(str(MASK_DIR / r['mask']), cv2.IMREAD_GRAYSCALE)
    mk_np  = cv2.resize(mk_raw, (512,512))
    mk01   = (mk_np > 127).astype(np.uint8)
    pr_pil = Image.fromarray(pr_np)
    metric_rows.append({
        'stem': r['stem'], 'occlusion_ratio': r['occlusion_ratio'],
        'bin_label': r.get('bin_label','?'),
        'l1': pixel_l1(gt_np,pr_np,mk01), 'l2': pixel_l2(gt_np,pr_np,mk01),
        'icp': compute_icp(pr_pil,mk_np), 'ss': compute_ss(pr_pil,mk01),
        'psnr': masked_psnr(gt_np,pr_np,mk01), 'ssim': masked_ssim(gt_np,pr_np,mk01),
        'lpips': masked_lpips(gt_np,pr_np,mk01),
    })
metric_df = pd.DataFrame(metric_rows)

print('Computing FID ...')
fid_value = cleanfid.compute_fid(str(EVAL_GT_DIR), str(EVAL_PRED_DIR), mode='clean')
print(f'FID = {fid_value:.2f}')
metric_df.to_csv(REPORT_DIR / 'metrics_per_image.csv', index=False)
print(f'Overall ({len(metric_df):,} images):')
print(metric_df[['l1','l2','icp','ss','psnr','ssim','lpips']].describe().round(4))


In [ ]:
# CELL 14 — Report: overall + per-bin + so sanh full-eval + paper
ALL_METRICS = ['l1','l2','icp','ss','psnr','ssim','lpips']

def bin_summary(df, lo, hi, label):
    sub = df[(df['occlusion_ratio']>=lo)&(df['occlusion_ratio']<hi)]
    if len(sub)==0: return {'bin':label,'n':0,**{m:None for m in ALL_METRICS}}
    row = {'bin':label,'n':len(sub)}
    for m in ALL_METRICS: row[m]=round(float(sub[m].mean()),4)
    return row

overall = {'bin':'overall','n':len(metric_df)}
for m in ALL_METRICS: overall[m]=round(float(metric_df[m].mean()),4)

rows = [overall]
for (lo,hi),lbl in zip(BIN_EDGES,BIN_LABELS):
    rows.append(bin_summary(metric_df,lo,hi,lbl))
summary_df = pd.DataFrame(rows)
summary_df.insert(summary_df.columns.get_loc('psnr')+3,'fid',
                  [round(fid_value,2)]+[None]*len(BIN_EDGES))

print('='*80)
print(f'RESULTS -- SD1.5 + LoRA r={LORA_RANK} + ControlNet + IP-Adapter')
print(f'cn_scale={BEST_CN}  ip_scale={BEST_IP}  steps={NUM_STEPS}  n={len(metric_df):,}')
print('='*80)
print(summary_df.to_string(index=False))
print('='*80)

# So sanh voi full-eval (SD1.5 + CN + IPA, khong LoRA)
baseline_file = PROJECT_ROOT / 'outputs' / 'lora_cn_ipa' / 'reports' / 'baseline_summary.csv'
if baseline_file.exists():
    bdf = pd.read_csv(baseline_file)
    base_row = bdf[bdf['bin']=='overall'].iloc[0]
    baseline_vals = {m: float(base_row[m]) for m in ALL_METRICS if m in base_row}
    print('(Baseline loaded from file)')
else:
    # Ket qua tu full-eval.ipynb chay tren Kaggle T4
    baseline_vals = {'l1':0.1438,'l2':0.0521,'icp':0.6081,'ss':0.3319,
                     'psnr':13.5475,'ssim':0.4151,'lpips':0.1285}
    print('(Baseline tu full-eval -- hardcoded)')

print('\n'+'='*80)
print('SO SANH: SD1.5+CN+IPA (no LoRA) vs SD1.5+LoRA+CN+IPA')
print('='*80)
cmp_rows = []
for name, vals in [('SD1.5+CN+IPA (no LoRA)', baseline_vals),
                    (f'SD1.5+LoRA r={LORA_RANK}+CN+IPA', {m:overall.get(m) for m in ALL_METRICS})]:
    row = {'config': name}
    row.update({m: round(vals.get(m,0),4) for m in ALL_METRICS})
    cmp_rows.append(row)
cmp_df = pd.DataFrame(cmp_rows)
print(cmp_df.to_string(index=False))

delta_row = {}
for m in ALL_METRICS:
    diff = (overall.get(m) or 0) - (baseline_vals.get(m) or 0)
    sign = chr(8593) if (m in ['icp','ss','psnr','ssim'] and diff>0) or (m in ['l1','l2','lpips'] and diff<0) else chr(8595)
    delta_row[m] = f'{diff:+.4f}{sign}'
print('Delta (LoRA - base):', delta_row)

print('\n'+'='*80)
print('SO SANH VOI Yan et al. (ICCV 2019) -- Synthetic M^gt')
print('='*80)
paper_rows = [
    {'method':'Deepfill [50]','l1':0.0284,'l2':0.0107,'icp':0.5620,'ss':0.8295},
    {'method':'Liu et al. [27]','l1':0.0272,'l2':0.0074,'icp':0.6284,'ss':0.8672},
    {'method':'Pathak et al. [35]','l1':0.0207,'l2':0.0088,'icp':0.5708,'ss':0.8517},
    {'method':'pix2pix [20]','l1':0.0174,'l2':0.0060,'icp':0.7081,'ss':0.9410},
    {'method':'SeGAN [12]','l1':0.0181,'l2':0.0055,'icp':0.6662,'ss':0.9371},
    {'method':'Yan et al. (best)','l1':0.0158,'l2':0.0038,'icp':0.7436,'ss':0.9458},
    {'method':'>>> Ours CN+IPA','l1':baseline_vals.get('l1'),'l2':baseline_vals.get('l2'),
     'icp':baseline_vals.get('icp'),'ss':baseline_vals.get('ss')},
    {'method':f'>>> Ours LoRA r={LORA_RANK}+CN+IPA','l1':overall.get('l1'),'l2':overall.get('l2'),
     'icp':overall.get('icp'),'ss':overall.get('ss')},
]
paper_df = pd.DataFrame(paper_rows)
pd.set_option('display.float_format', '{:.4f}'.format)
print(paper_df.to_string(index=False))
print('='*80+'\nL1 down  L2 down  ICP up  SS up')

summary_df.to_csv(REPORT_DIR/'summary_table.csv', index=False)
paper_df.to_csv(REPORT_DIR/'paper_comparison.csv', index=False)
cmp_df.to_csv(REPORT_DIR/'lora_vs_baseline.csv', index=False)
with open(REPORT_DIR/'eval_config.json','w') as f:
    json.dump({'config':f'SD1.5+LoRA r={LORA_RANK}+CN+IPA','lora_ckpt':str(best_ckpt),
               'cn_scale':BEST_CN,'ip_scale':BEST_IP,'num_steps':NUM_STEPS,'n':int(len(metric_df)),
               **{m:overall.get(m) for m in ALL_METRICS},'fid':round(fid_value,2)}, f, indent=2)
print('\nFiles saved:', [f.name for f in sorted(REPORT_DIR.glob('*'))])


In [ ]:
# CELL 15 — Visualization
import matplotlib.pyplot as plt

# 15a. Bar charts theo bin
plot_df = summary_df[summary_df['bin'] != 'overall'].copy()
fig, axes = plt.subplots(1, 3, figsize=(14,4))
for ax,col,label in [(axes[0],'psnr','PSNR up (dB)'),(axes[1],'ssim','SSIM up'),(axes[2],'lpips','LPIPS down')]:
    colors=['#3498db','#2ecc71','#e74c3c']
    bars=ax.bar(plot_df['bin'],plot_df[col],color=colors,edgecolor='white',width=0.5)
    ax.set_title(label,fontsize=11,fontweight='bold'); ax.set_xlabel('Occlusion level')
    for bar,v in zip(bars,plot_df[col]):
        if v is not None:
            ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.002,
                    f'{v:.3f}',ha='center',va='bottom',fontsize=9)
plt.suptitle(f'Metrics by bin -- LoRA r={LORA_RANK} + CN + IPA', y=1.02)
plt.tight_layout(); plt.show()

# 15b. PSNR scatter
fig2,ax2=plt.subplots(figsize=(8,4))
sc=ax2.scatter(metric_df['occlusion_ratio'],metric_df['psnr'],
               c=metric_df['psnr'],cmap='RdYlGn',alpha=0.4,s=8)
plt.colorbar(sc,ax=ax2,label='PSNR (dB)')
for (lo,hi) in BIN_EDGES: ax2.axvline(lo,color='gray',linestyle='--',linewidth=0.8)
ax2.axvline(BIN_EDGES[-1][1],color='gray',linestyle='--',linewidth=0.8)
ax2.set_xlabel('Occlusion ratio'); ax2.set_ylabel('PSNR (dB)')
ax2.set_title('PSNR vs occlusion ratio')
plt.tight_layout(); plt.show()

# 15c. Visual grid: best + worst theo PSNR moi bin
fig4,axes4=plt.subplots(len(BIN_EDGES)*2,5,figsize=(18,5*len(BIN_EDGES)*2))
row_idx=0
for (lo,hi),label in zip(BIN_EDGES,BIN_LABELS):
    bin_m=metric_df[(metric_df['occlusion_ratio']>=lo)&(metric_df['occlusion_ratio']<hi)]
    if len(bin_m)==0: continue
    best_stem=bin_m.loc[bin_m['psnr'].idxmax(),'stem']
    worst_stem=bin_m.loc[bin_m['psnr'].idxmin(),'stem']
    for stem,tag in [(best_stem,'BEST'),(worst_stem,'WORST')]:
        row_m=pred_df[pred_df['stem']==stem].iloc[0]
        m_row=bin_m[bin_m['stem']==stem].iloc[0]
        imgs=[
            (Image.open(GT_DIR  /row_m['x_gt']).convert('RGB'),'x_gt'),
            (Image.open(OCC_DIR /row_m['x_occ']).convert('RGB'),'x_occ'),
            (Image.open(MASK_DIR/row_m['mask']).convert('L'),'Mask'),
            (extract_canny_masked(
                Image.open(OCC_DIR /row_m['x_occ']).convert('RGB'),
                Image.open(MASK_DIR/row_m['mask']).convert('L')),'Canny'),
            (Image.open(PRED_DIR/row_m['pred']).convert('RGB'),'x_hat'),
        ]
        for col_j,(img,title) in enumerate(imgs):
            ax=axes4[row_idx,col_j]
            ax.imshow(img,cmap='gray' if title=='Mask' else None)
            if row_idx==0: ax.set_title(title,fontsize=9,fontweight='bold')
            ax.axis('off')
        axes4[row_idx,0].set_ylabel(
            f'Bin {label}\n{tag}\nPSNR={m_row["psnr"]:.1f}  SSIM={m_row["ssim"]:.3f}\nLPIPS={m_row["lpips"]:.3f}',
            fontsize=7,labelpad=4)
        row_idx+=1
for ax in axes4[row_idx:].flatten(): ax.axis('off')
plt.suptitle(f'Best & Worst per bin -- LoRA r={LORA_RANK} + CN + IPA',y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# CELL 16 — VRAM Report + Zip
import zipfile, subprocess

if torch.cuda.is_available():
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    props   = torch.cuda.get_device_properties(0)
    print(f'GPU        : {props.name}')
    print(f'Total VRAM : {props.total_memory/1024**3:.1f} GiB')
    print(f'Peak alloc : {peak_mb:.0f} MiB ({peak_mb/1024:.2f} GiB)')
    print(f'xFormers   : {xformers_ok}  |  IP-Adapter: {use_ip}')
    try:
        smi=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,memory.used,memory.free',
            '--format=csv,noheader,nounits'],stderr=subprocess.DEVNULL).decode().strip()
        print('nvidia-smi :', smi)
    except Exception: pass

zip_path = OUT_BASE / 'lora_cn_ipa_reports.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=3) as zf:
    for f in sorted(REPORT_DIR.glob('*')):
        zf.write(f, f'reports/{f.name}')
    curves_f = LORA_OUT / 'training_curves.png'
    if curves_f.exists(): zf.write(curves_f, 'training_curves.png')
print(f'Zip: {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)')
print('Contains:', [f.name for f in sorted(REPORT_DIR.glob('*'))])
